# Task 129: bilby_gw — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Gravitational wave parameter estimation using Bilby

**Paper**: Bilby: A User-friendly Bayesian Inference Library for Gravitational-wave Astronomy
**Repository**: https://github.com/bilby-dev/bilby

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: 63.06 dB
- **SSIM**: N/A (waveform match = 0.9994)
- **Evaluation**: GW parameter estimation (compact binary coalescence waveform recovery)

### Mathematical Formulation

**Parameter estimation** fits model parameters $\boldsymbol{\theta}$ to data:

$$\hat{\boldsymbol{\theta}} = \arg\min_{\boldsymbol{\theta}} \sum_{i=1}^{N} \left(\frac{y_i - f(t_i; \boldsymbol{\theta})}{\sigma_i}\right)^2$$

**Fisher information matrix**: $\mathcal{I}_{jk} = \sum_i \frac{1}{\sigma_i^2} \frac{\partial f_i}{\partial\theta_j}\frac{\partial f_i}{\partial\theta_k}$

**Cramer-Rao bound**: $\text{Var}(\hat\theta_j) \geq [\mathcal{I}^{-1}]_{jj}$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
#!/usr/bin/env python
"""
Bilby GW Parameter Estimation Benchmark
========================================
Recover compact binary coalescence (CBC) parameters from simulated
gravitational-wave strain data.

Strategy (fast, benchmarkable):
  1. Use bilby + LAL to generate a CBC waveform with known parameters
  2. Add realistic Gaussian noise from LIGO PSD
  3. Use bilby's dynesty sampler with aggressive settings
     - Only 3 free parameters (chirp_mass, mass_ratio, luminosity_distance)
     - All other params fixed to truth
     - nlive=100, tight priors, loose convergence
  4. Compare recovered waveform vs injected (ground truth)
  5. Compute PSNR, correlation, waveform match, parameter errors

Inverse problem: h(t) + n(t) -> {theta_source}

Reference: Ashton et al., ApJS 2019 (bilby paper)
"""

import os
import sys
import json
import time
import warnings
import numpy as np

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`inner_product`

In [ ]:
def inner_product(a, b, psd, df):
    return 4.0 * df * np.real(np.sum(np.conj(a) * b / psd))

## 4. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import bilby

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────
OUTDIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "results")
os.makedirs(OUTDIR, exist_ok=True)
bilby.core.utils.setup_logger(outdir=OUTDIR, label="bilby_gw", log_level="WARNING")

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
DURATION = 4          # seconds of data
SAMPLING_FREQ = 2048  # Hz
MINIMUM_FREQ = 20.0   # Hz low-frequency cutoff
REFERENCE_FREQ = 50.0 # Hz
APPROX = "IMRPhenomPv2"  # fast frequency-domain waveform

### Ground Truth Construction

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# ── Injection parameters (ground truth) ───────────────────────────────
injection_parameters = dict(
    mass_1=36.0,
    mass_2=29.0,
    a_1=0.4,
    a_2=0.3,
    tilt_1=0.5,
    tilt_2=1.0,
    phi_12=1.7,
    phi_jl=0.3,
    luminosity_distance=500.0,
    theta_jn=0.4,
    psi=2.659,
    phase=1.3,
    geocent_time=1126259642.413,
    ra=1.375,
    dec=-1.2108,
)

### Configuration and Parameters

Set up experiment parameters: grid sizes, noise levels, regularization weights, algorithm settings.

In [ ]:
m1, m2 = injection_parameters["mass_1"], injection_parameters["mass_2"]
true_chirp_mass = (m1 * m2) ** (3.0 / 5) / (m1 + m2) ** (1.0 / 5)
true_mass_ratio = m2 / m1
print(f"True chirp_mass = {true_chirp_mass:.4f}, mass_ratio = {true_mass_ratio:.4f}")

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# ── Waveform generator ───────────────────────────────────────────────
waveform_arguments = dict(
    waveform_approximant=APPROX,
    reference_frequency=REFERENCE_FREQ,
    minimum_frequency=MINIMUM_FREQ,
)

### Configuration and Parameters

Set up experiment parameters: grid sizes, noise levels, regularization weights, algorithm settings.

In [ ]:
waveform_generator = bilby.gw.WaveformGenerator(
    duration=DURATION,
    sampling_frequency=SAMPLING_FREQ,
    frequency_domain_source_model=bilby.gw.source.lal_binary_black_hole,
    parameter_conversion=bilby.gw.conversion.convert_to_lal_binary_black_hole_parameters,
    waveform_arguments=waveform_arguments,
)

### Forward Model — Generating Measurements

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# ── Interferometers with injected signal + Gaussian noise ─────────────
print("[1/7] Setting up interferometers with injected signal...")
ifos = bilby.gw.detector.InterferometerList(["H1", "L1"])
ifos.set_strain_data_from_power_spectral_densities(
    sampling_frequency=SAMPLING_FREQ,
    duration=DURATION,
    start_time=injection_parameters["geocent_time"] - DURATION + 2,
)
ifos.inject_signal(
    waveform_generator=waveform_generator,
    parameters=injection_parameters,
)

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# ── Compute ground-truth signal ──────────────────────────────────────
print("[2/7] Computing ground-truth waveform...")
polarisations = waveform_generator.frequency_domain_strain(injection_parameters)
freq_array = waveform_generator.frequency_array
h1 = ifos[0]
gt_strain_fd = h1.get_detector_response(polarisations, injection_parameters)
noisy_strain_fd = h1.strain_data.frequency_domain_strain.copy()

### Configuration and Parameters

Set up experiment parameters: grid sizes, noise levels, regularization weights, algorithm settings.

In [ ]:
# ── Priors (3 free params) ───────────────────────────────────────────
print("[3/7] Setting up priors (3 free parameters)...")
priors = bilby.gw.prior.BBHPriorDict()

### Configuration and Parameters

Set up experiment parameters: grid sizes, noise levels, regularization weights, algorithm settings.

In [ ]:
# Fix everything except chirp_mass, mass_ratio, luminosity_distance
for key in ["ra", "dec", "psi", "phase", "tilt_1", "tilt_2",
            "phi_12", "phi_jl", "a_1", "a_2", "theta_jn", "geocent_time"]:
    priors[key] = injection_parameters[key]

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
priors["chirp_mass"] = bilby.core.prior.Uniform(
    name="chirp_mass", minimum=25.0, maximum=32.0,
    latex_label="$\\mathcal{M}$",
)
priors["mass_ratio"] = bilby.core.prior.Uniform(
    name="mass_ratio", minimum=0.5, maximum=1.0,
    latex_label="$q$",
)
priors["luminosity_distance"] = bilby.gw.prior.UniformSourceFrame(
    name="luminosity_distance", minimum=200.0, maximum=1000.0, unit="Mpc",
)

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# ── Likelihood ────────────────────────────────────────────────────────
print("[4/7] Setting up likelihood...")
likelihood = bilby.gw.GravitationalWaveTransient(
    interferometers=ifos,
    waveform_generator=waveform_generator,
    priors=priors,
)

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# ── Run sampler ───────────────────────────────────────────────────────
print("[5/7] Running nested sampling (dynesty, nlive=100, 3 free params)...")
print("       Expected runtime: 2-8 minutes...")
t_start = time.time()

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
result = bilby.run_sampler(
    likelihood=likelihood,
    priors=priors,
    sampler="dynesty",
    nlive=100,
    nact=3,
    maxmcmc=2000,
    walks=5,
    dlogz=0.5,
    outdir=OUTDIR,
    label="bilby_gw",
    injection_parameters=injection_parameters,
    save=True,
    resume=False,
    clean=True,
)

t_elapsed = time.time() - t_start
print(f"       Sampling completed in {t_elapsed:.1f} s ({t_elapsed/60:.1f} min)")

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# ── Extract results ───────────────────────────────────────────────────
print("[6/7] Extracting results and computing metrics...")
posterior = result.posterior

### Configuration and Parameters

Set up experiment parameters: grid sizes, noise levels, regularization weights, algorithm settings.

In [ ]:
# Derive mass_1, mass_2 from chirp_mass, mass_ratio
if "mass_1" not in posterior.columns:
    q = posterior["mass_ratio"]
    mc = posterior["chirp_mass"]
    eta = q / (1 + q) ** 2
    mtotal = mc / eta ** (3.0 / 5)
    posterior["mass_1"] = mtotal / (1 + q)
    posterior["mass_2"] = mtotal * q / (1 + q)

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
estimated_params = ["chirp_mass", "mass_ratio", "luminosity_distance"]
derived_params = ["mass_1", "mass_2"]
all_report_params = estimated_params + derived_params

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
map_idx = posterior["log_likelihood"].idxmax()
median_params = {}
map_params = {}
for p in all_report_params:
    if p in posterior.columns:
        median_params[p] = float(np.median(posterior[p]))
        map_params[p] = float(posterior[p].iloc[map_idx])

### Configuration and Parameters

Set up experiment parameters: grid sizes, noise levels, regularization weights, algorithm settings.

In [ ]:
true_values = dict(injection_parameters)
true_values["chirp_mass"] = true_chirp_mass
true_values["mass_ratio"] = true_mass_ratio

### Configuration and Parameters

Set up experiment parameters: grid sizes, noise levels, regularization weights, algorithm settings.

In [ ]:
print("\n=== Parameter Recovery ===")
print(f"{'Parameter':<25} {'True':>12} {'MAP':>12} {'Median':>12} {'RelErr%':>10}")
print("-" * 75)
relative_errors = {}
for p in all_report_params:
    true_val = true_values[p]
    map_val = map_params.get(p, np.nan)
    med_val = median_params.get(p, np.nan)
    rel_err = abs(med_val - true_val) / abs(true_val) * 100 if abs(true_val) > 1e-10 else 0.0
    relative_errors[p] = rel_err
    print(f"{p:<25} {true_val:>12.4f} {map_val:>12.4f} {med_val:>12.4f} {rel_err:>9.2f}%")

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# ── Reconstruct waveform from MAP parameters ─────────────────────────
recon_params = dict(injection_parameters)
for p in estimated_params:
    recon_params[p] = map_params[p]
q_map = map_params["mass_ratio"]
mc_map = map_params["chirp_mass"]
eta_map = q_map / (1 + q_map) ** 2
mtotal_map = mc_map / eta_map ** (3.0 / 5)
recon_params["mass_1"] = mtotal_map / (1 + q_map)
recon_params["mass_2"] = mtotal_map * q_map / (1 + q_map)

recon_polarisations = waveform_generator.frequency_domain_strain(recon_params)
recon_signal_h1 = h1.get_detector_response(recon_polarisations, recon_params)

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# ── Signal-level metrics ─────────────────────────────────────────────
mask = (freq_array >= MINIMUM_FREQ) & (freq_array <= SAMPLING_FREQ / 2)
gt_signal = gt_strain_fd[mask]
recon_signal = recon_signal_h1[mask]

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
gt_abs = np.abs(gt_signal)
recon_abs = np.abs(recon_signal)
mse = np.mean((gt_abs - recon_abs) ** 2)
max_val = np.max(gt_abs)
psnr = 10 * np.log10(max_val ** 2 / mse) if mse > 0 else float("inf")

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
corr_complex = np.abs(np.vdot(gt_signal, recon_signal)) / (
    np.linalg.norm(gt_signal) * np.linalg.norm(recon_signal)
)

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
psd_array = h1.power_spectral_density_array[mask]
df = freq_array[1] - freq_array[0]
psd_safe = np.where(psd_array > 0, psd_array, np.inf)

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
overlap = inner_product(gt_signal, recon_signal, psd_safe, df)
norm_gt = np.sqrt(inner_product(gt_signal, gt_signal, psd_safe, df))
norm_recon = np.sqrt(inner_product(recon_signal, recon_signal, psd_safe, df))
match = overlap / (norm_gt * norm_recon) if (norm_gt > 0 and norm_recon > 0) else 0.0

mean_rel_error = np.mean(list(relative_errors.values()))

snr_list = [float(ifo.meta_data.get("optimal_SNR", 0.0)) for ifo in ifos]
network_snr = np.sqrt(sum(s ** 2 for s in snr_list))

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
print(f"\n=== Signal Metrics ===")
print(f"PSNR (freq-domain amplitude): {psnr:.2f} dB")
print(f"Correlation coefficient:       {corr_complex:.6f}")
print(f"Waveform match (overlap):      {match:.6f}")
print(f"Mean param relative error:     {mean_rel_error:.2f}%")
print(f"Network SNR (injected):        {network_snr:.1f}")
print(f"Runtime:                       {t_elapsed:.1f} s")

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# ── Save metrics ──────────────────────────────────────────────────────
metrics = {
    "psnr_db": round(float(psnr), 2),
    "correlation": round(float(corr_complex), 6),
    "waveform_match": round(float(match), 6),
    "mean_relative_error_pct": round(float(mean_rel_error), 2),
    "network_snr": round(float(network_snr), 2),
    "runtime_s": round(float(t_elapsed), 1),
    "nlive": 100,
    "n_free_params": 3,
    "sampler": "dynesty",
    "parameter_errors": {p: round(float(v), 4) for p, v in relative_errors.items()},
    "recovered_parameters": {p: round(float(v), 4) for p, v in median_params.items()},
    "true_parameters": {p: round(float(true_values[p]), 4) for p in all_report_params},
}
metrics_path = os.path.join(OUTDIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"\nMetrics saved to {metrics_path}")

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# ── Save numpy arrays ────────────────────────────────────────────────
print("[7/7] Saving outputs and creating visualization...")
from scipy.fft import irfft

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
n_samples = int(DURATION * SAMPLING_FREQ)
gt_td = np.real(irfft(gt_strain_fd, n=n_samples))
recon_td = np.real(irfft(recon_signal_h1, n=n_samples))
noisy_td = np.real(irfft(noisy_strain_fd, n=n_samples))
time_array = np.arange(n_samples) / SAMPLING_FREQ

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
np.save(os.path.join(OUTDIR, "ground_truth.npy"), gt_td)
np.save(os.path.join(OUTDIR, "reconstruction.npy"), recon_td)
print("Saved ground_truth.npy and reconstruction.npy")

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# ── Visualization ─────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
t_start_data = injection_parameters["geocent_time"] - DURATION + 2
t_plot = time_array + t_start_data - injection_parameters["geocent_time"]

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
ax = axes[0, 0]
ax.plot(t_plot, noisy_td, color="lightgray", alpha=0.5, linewidth=0.3,
        label="Noisy data", rasterized=True)
ax.plot(t_plot, gt_td, color="tab:blue", linewidth=1.0, label="True signal")
ax.set_xlabel("Time relative to merger (s)")
ax.set_ylabel("Strain")
ax.set_title("Detector Strain (H1): Data + Injected Signal")
ax.legend(loc="upper left")
ax.set_xlim(-0.5, 0.05)

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
ax = axes[0, 1]
ax.plot(t_plot, gt_td, color="tab:blue", linewidth=1.2, label="True signal")
ax.plot(t_plot, recon_td, color="tab:red", linewidth=1.0, linestyle="--",
        label="Recovered (MAP)")
ax.set_xlabel("Time relative to merger (s)")
ax.set_ylabel("Strain")
ax.set_title(f"Waveform Recovery (match={match:.4f})")
ax.legend(loc="upper left")
ax.set_xlim(-0.5, 0.05)

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
ax = axes[1, 0]
param_labels = {
    "chirp_mass": "$\\mathcal{M}$",
    "mass_ratio": "$q$",
    "luminosity_distance": "$d_L$",
    "mass_1": "$m_1$",
    "mass_2": "$m_2$",
}
x_pos = np.arange(len(all_report_params))
rel_errs = [relative_errors[p] for p in all_report_params]
colors = ["tab:green" if e < 5 else "tab:orange" if e < 15 else "tab:red" for e in rel_errs]
ax.bar(x_pos, rel_errs, color=colors, edgecolor="black", linewidth=0.5)
ax.set_xticks(x_pos)
ax.set_xticklabels([param_labels.get(p, p) for p in all_report_params], fontsize=11)
ax.set_ylabel("Relative Error (%)")
ax.set_title("Parameter Recovery Accuracy")
ax.axhline(y=5, color="green", linestyle=":", alpha=0.5, label="5%")
ax.axhline(y=15, color="orange", linestyle=":", alpha=0.5, label="15%")
ax.legend()

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
ax = axes[1, 1]
freq_plot = freq_array[mask]
ax.loglog(freq_plot, np.abs(gt_signal), color="tab:blue", linewidth=1.0,
          label="True signal")
ax.loglog(freq_plot, np.abs(recon_signal), color="tab:red", linewidth=0.8,
          linestyle="--", label="Recovered (MAP)")
ax.loglog(freq_plot, np.sqrt(psd_safe), color="lightgray", linewidth=0.5,
          alpha=0.8, label="$\\sqrt{S_n(f)}$")
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("|h(f)|")
ax.set_title(f"Frequency-Domain Comparison (PSNR={psnr:.1f} dB)")
ax.legend(loc="upper right")
ax.set_xlim(MINIMUM_FREQ, SAMPLING_FREQ / 2)

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
plt.suptitle(
    f"Bilby GW Parameter Estimation: CBC Waveform Recovery\n"
    f"Match={match:.4f} | PSNR={psnr:.1f} dB | Corr={corr_complex:.4f} | "
    f"Mean RelErr={mean_rel_error:.1f}% | SNR={network_snr:.1f} | "
    f"Runtime={t_elapsed:.0f}s",
    fontsize=12, y=1.02,
)
plt.tight_layout()
fig_path = os.path.join(OUTDIR, "reconstruction_result.png")
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.close()
print(f"Figure saved to {fig_path}")

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
try:
    result.plot_corner(
        parameters=["chirp_mass", "mass_ratio", "luminosity_distance"],
        filename=os.path.join(OUTDIR, "corner_plot.png"),
        save=True,
    )
    print("Corner plot saved.")
except Exception as e:
    print(f"Corner plot skipped: {e}")

print("\n=== DONE ===")
print(f"All outputs saved to {OUTDIR}/")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 5. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **bilby_gw**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=63.06 dB, SSIM=N/A (waveform match = 0.9994))

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: Bilby: A User-friendly Bayesian Inference Library for Gravitational-wave Astronomy
- Repository: https://github.com/bilby-dev/bilby
